# 1. Data Understanding — Credit Risk Prediction

## Business Objective
Loan investors and financial institutions need to assess the likelihood that a borrower
will default before capital is committed to a loan. This project uses LendingClub's
historical loan data to build a model that estimates the probability of default, and to
surface the borrower and loan characteristics most associated with credit risk.

## Business Question
Given borrower and loan characteristics, what is the probability that a loan will end in
default (Charged Off / Default) rather than being repaid in full (Fully Paid)?

## Intended Use / Persona
This model is built for an **investor-facing use case**: assessing risk on loans that
have already been listed and graded by LendingClub, to support funding decisions among
already-available loans. (Note: this is a deliberate scope decision — a separate
"pure borrower" model, excluding LendingClub's own assigned grade/rate, is trained and
compared in notebook 4 for the alternative use case of screening a brand-new applicant
who has not yet been graded.)

## Dataset
- **Source:** LendingClub loan data, publicly released historical dataset (Kaggle mirror)
- **Grain:** One row per originated loan
- **Scale:** 2,260,701 rows, 151 raw columns, ~2.5GB in memory as loaded
- **Coverage:** Loans issued across multiple years (exact range confirmed in Section 8)
- **Target concept:** `loan_status` — will be converted into a binary default/no-default
  target in preprocessing (notebook 3), restricted to loans with a completed outcome

## What this notebook covers
Structural profiling of the raw data only — no cleaning, filtering, or feature engineering
happens here. Every finding below is the documented justification for decisions made in
`3_data_preprocessing.ipynb`.

In [1]:
#---------------------------------------
# Setup
#---------------------------------------
import os
import sys
import pandas as pd
import numpy as np

def find_project_root(marker="config.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not locate project root (looked for '{marker}')")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

pd.set_option("display.max_columns", 100)
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\venut\OneDrive\Desktop\credit_risk_project


## 1. Load raw data

In [2]:
def load_full_dataset_memory_safe(path, start_chunksize=100_000, min_chunksize=10_000):
    """
    Loads the full raw file safely on memory-constrained machines. Reads in
    chunks and downcasts dtypes as it goes (float64 -> float32, int64 ->
    smallest safe int type, low-cardinality object columns -> category) to
    reduce peak memory usage. If it still runs out of memory, it automatically
    retries with a smaller chunk size instead of failing outright.
    """
    chunksize = start_chunksize
    while chunksize >= min_chunksize:
        try:
            chunks = []
            for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
                for col in chunk.select_dtypes(include=["float64"]).columns:
                    chunk[col] = pd.to_numeric(chunk[col], downcast="float")
                for col in chunk.select_dtypes(include=["int64"]).columns:
                    chunk[col] = pd.to_numeric(chunk[col], downcast="integer")
                for col in chunk.select_dtypes(include=["object"]).columns:
                    if chunk[col].nunique(dropna=True) / max(len(chunk), 1) < 0.5:
                        chunk[col] = chunk[col].astype("category")
                chunks.append(chunk)
            return pd.concat(chunks, ignore_index=True)
        except (MemoryError, pd.errors.ParserError) as e:
            print(f"Load failed at chunksize={chunksize} ({type(e).__name__}). "
                  f"Retrying with a smaller chunksize...")
            chunksize = chunksize // 2
    raise MemoryError(
        f"Could not load the file even at chunksize={min_chunksize}. "
        f"Close other programs to free RAM and try again, or reduce min_chunksize."
    )

df = load_full_dataset_memory_safe("data/raw/dataset.csv")
print("Shape:", df.shape)
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Shape: (2260701, 151)
Memory usage: 4386.2 MB


**Result: 2,260,701 rows × 151 columns.** This is the full, unfiltered raw export —
every downstream row/column count in this project traces back to this number.

## 2. Structural overview

In [3]:
print("Columns:", len(df.columns))
df.info(verbose=False, show_counts=True)

Columns: 151
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: category(9), float32(100), float64(13), object(29)
memory usage: 1.6+ GB


**Result: 151 columns — 113 float64, 38 object.** Memory footprint is ~2.5GB loaded
uncompressed, which is why preprocessing and business-insights notebooks load only a
required-column subset via chunked reads rather than the full file.

In [4]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,...,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.990000,123.029999,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.910000,0.0,Aug-2003,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.700001,13.0,w,0.00,0.00,4421.723917,4421.72,3600.00,821.72,0.0,0.0,0.0,Jan-2019,122.67,NaN,...,4.0,7.0,0.0,0.0,0.0,3.0,76.900002,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.990000,820.280029,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.059999,1.0,Dec-1999,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.200001,38.0,w,0.00,0.00,25679.660000,25679.66,24700.00,979.66,0.0,0.0,0.0,Jun-2016,926.35,NaN,...,5.0,22.0,0.0,0.0,0.0,2.0,97.400002,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.780000,432.660004,B,B4,truck driver,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,home_improvement,NaN,605xx,IL,10.780000,0.0,Aug-2000,695.0,699.0,0.0,NaN,NaN,6.0,0.0,7869.0,56.200001,18.0,w,0.00,0.00,22705.924294,22705.92,20000.00,2705.92,0.0,0.0,0.0,Jun-2017,15813.30,NaN,...,3.0,6.0,0.0,0.0,0.0,0.0,100.000000,50.0,0.0,0.0,218418.0,18696.0,6200.0,14877.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.850000,829.900024,C,C5,Information Systems Officer,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,Current,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,076xx,NJ,17.059999,0.0,Sep-2008,785.0,789.0,0.0,NaN,NaN,13.0,0.0,7802.0,11.600000,17.0,w,15897.65,15897.65,31464.010000,31464.01,19102.35,12361.66,0.0,0.0,0.0,Feb-2019,829.90,Apr-2019,...,5.0,13.0,0.0,0.0,0.0,1.0,100.000000,0.0,0.0,0.0,381215.0,52226.0,62500.0,18

## 3. `id` column dtype check
`id` displays as numeric-looking values (e.g. `68407277`) but loads as `object`, not
an integer type. Investigating why, since an unexplained dtype mismatch on an
identifier column should not be left unresolved.

In [5]:
print("id dtype:", df["id"].dtype)
print("Sample values:", df["id"].head(5).tolist())

# Check for non-numeric characters that would force object dtype
non_numeric_ids = df[~df["id"].astype(str).str.match(r"^\d+$", na=False)]
print(f"Non-purely-numeric id values: {len(non_numeric_ids)}")
if len(non_numeric_ids) > 0:
    print(non_numeric_ids["id"].head(10).tolist())

id dtype: object
Sample values: [68407277, 68355089, 68341763, 66310712, 68476807]
Non-purely-numeric id values: 33
['Total amount funded in policy code 1: 6417608175', 'Total amount funded in policy code 2: 1944088810', 'Total amount funded in policy code 1: 1741781700', 'Total amount funded in policy code 2: 564202131', 'Total amount funded in policy code 1: 1791201400', 'Total amount funded in policy code 2: 651669342', 'Total amount funded in policy code 1: 1443412975', 'Total amount funded in policy code 2: 511988838', 'Total amount funded in policy code 1: 2063142975', 'Total amount funded in policy code 2: 823319310']


**Decision:** if non-numeric values are confirmed above, `id` should be kept as a
string/object identifier throughout (never cast to numeric or used as a model feature —
it's a row identifier only). If no non-numeric values are found, the object dtype is
most likely due to mixed formatting on load and can be safely cast to int64 in
preprocessing without loss.

## 4. Duplicate check

In [6]:
dupe_rows = df.duplicated().sum()
print(f"Fully duplicate rows: {dupe_rows}")

dupe_ids = df["id"].duplicated().sum()
print(f"Duplicate 'id' values: {dupe_ids}")

Fully duplicate rows: 0
Duplicate 'id' values: 0


**Result: 0 fully duplicate rows.** No row-level deduplication is required.

## 5. Missing value analysis
Full column-level missing count and percentage, to separate three distinct categories
of missingness that require different handling — not all missing data is the same
problem.

In [7]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct.round(2)})
missing_df = missing_df[missing_df["Missing Count"] > 0]
print(f"Columns with at least one missing value: {len(missing_df)} / {len(df.columns)}")
missing_df.head(30)

Columns with at least one missing value: 150 / 151


,Missing Count,Missing %
member_id,2260701,100.00
orig_projected_additional_accrued_interest,2252050,99.62
hardship_reason,2249784,99.52
hardship_payoff_balance_amount,2249784,99.52
hardship_last_payment_amount,2249784,99.52
payment_plan_start_date,2249784,99.52
hardship_type,2249784,99.52
hardship_status,2249784,99.52
hardship_start_date,2249784,99.52
deferral_term,2249784,99.52


**Three distinct categories found here, each needing a different decision — documented
now so preprocessing isn't making these calls silently:**

**(a) `member_id` — 100.00% missing (2,260,701 / 2,260,701).**
Confirmed completely empty, not a loading artifact. LendingClub scrubs this field
entirely in the public release for borrower privacy. **Decision: drop.** It carries
zero information.

**(b) Hardship/settlement program columns — ~98.5%–99.6% missing**
(`hardship_reason`, `hardship_type`, `hardship_status`, `hardship_amount`,
`hardship_dpd`, `hardship_length`, `settlement_status`, `settlement_date`,
`settlement_amount`, `settlement_percentage`, `settlement_term`,
`debt_settlement_flag_date`, `sec_app_*` joint-applicant fields, and similar). These
are only populated for the small minority of borrowers who entered a hardship or debt
settlement program — a real, rare event, not a data quality failure. **Decision: drop
from the modeling feature set.** At <2% non-null, they carry too little signal to
reliably impute or use as predictors, and are themselves a downstream consequence of
distress rather than a pre-origination risk factor.

**(c) Core numeric predictors with low missingness** (`annual_inc`, `dti`,
`delinq_2yrs`, `revol_util`, etc., each well under 1% missing based on the counts
above). **Decision: retain and impute** (median for numeric, most-frequent for
categorical) — the missingness rate is low enough that imputation is a reasonable,
low-risk strategy. This is implemented inside the modeling pipeline in notebook 4
(post-split, to avoid leakage), not here.

## 6. Numeric summary
Includes two known data quality anomalies that were confirmed directly from this
output and must be addressed in preprocessing — flagging them here with evidence.

In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
member_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amnt,2260668.0,15046.931641,9190.246094,500.00,8000.000000,12900.000000,20000.000000,40000.000000
funded_amnt,2260668.0,15041.664062,9188.413086,500.00,8000.000000,12875.000000,20000.000000,40000.000000
funded_amnt_inv,2260668.0,15023.437745,9192.331679,0.00,8000.000000,12800.000000,20000.000000,40000.000000
int_rate,2260668.0,13.092829,4.832139,5.31,9.490000,12.620000,15.990000,30.990000
...,...,...,...,...,...,...,...,...
hardship_payoff_balance_amount,10917.0,11636.883942,7625.988281,55.73,5627.000000,10028.390000,16151.890000,40306.410000
hardship_last_payment_amount,10917.0,193.994324,198.629501,0.01,44.439999,133.160004,284.190002,1407.859985
settlement_amount,34246.0,5010.664267,3693.122590,44.21,2208.000000,4146.110000,6850.172500,33601.000000
settlement_percentage,34246.0,47.780361,7.311822,0.20,45.000000,45.000000,50.000000,521.349976


**Two anomalies confirmed directly in this table:**

**(a) `dti` has a minimum value of -1.0.** Debt-to-income ratio cannot legitimately be
negative — this is a LendingClub sentinel/placeholder value (used when DTI could not be
calculated), not a real observation. **Decision: this value must be treated as missing
(converted to NaN), not as a genuine low-DTI data point**, before any capping or
modeling. Left untreated, it would teach a model the opposite of the truth for these
rows.

**(b) `annual_inc` has a maximum value of $110,000,000.** This is almost certainly a
data entry error or an extreme, non-representative outlier — no realistic individual
borrower population has annual incomes reaching nine figures at meaningful frequency.
Left unaddressed, this single-column extreme value is very likely to distort any ratio
feature built from income (`loan_to_income`, `installment_to_income`), since a division
by a near-correct denominator produces a near-zero ratio while a handful of these
extreme outliers produce enormous, distorting ratio values. **Decision: cap or winsorize
`annual_inc` (e.g. at the 99th percentile) before any income-based ratio is engineered
in preprocessing** — this is a direct fix for an outlier problem observed later in the
pipeline, and the root cause is visible right here.

## 7. Categorical summary
`.describe()` alone only covers numeric columns — deliberately checking categorical
columns separately so nothing is silently skipped.

In [9]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns: {len(cat_cols)}")
df[cat_cols].describe().T

Categorical columns: 29


,count,unique,top,freq
id,2260701,2260701,68407277,1
emp_title,2093699,512694,Teacher,38824
home_ownership,2260668,6,MORTGAGE,1111450
issue_d,2260668,139,Mar-2016,61992
loan_status,2260668,9,Fully Paid,1076751
pymnt_plan,2260668,2,n,2260048
url,2260668,2260668,https://lendingclub.com/browse/loanDetail.acti...,1
desc,126065,124500,,252
purpose,2260668,14,debt_consolidation,1277877
title,2237342,63154,Debt consolidation,1153293


In [10]:
# Cardinality — informs one-hot encoding decisions in preprocessing.
# High-cardinality columns may need different treatment than simple one-hot encoding.
cat_cardinality = df[cat_cols].nunique().sort_values(ascending=False)
cat_cardinality.head(20)

id                           2260701
url                          2260668
emp_title                     512694
desc                          124500
title                          63154
zip_code                         956
earliest_cr_line                 754
sec_app_earliest_cr_line         663
last_credit_pull_d               141
issue_d                          139
last_pymnt_d                     136
next_pymnt_d                     106
settlement_date                   90
debt_settlement_flag_date         83
addr_state                        51
hardship_end_date                 28
hardship_start_date               27
payment_plan_start_date           27
purpose                           14
loan_status                        9
dtype: int64

## 8. Target variable: `loan_status`
Full breakdown of every status value, before any filtering decision is made.

In [11]:
status_counts = df["loan_status"].value_counts()
status_pct = (status_counts / len(df) * 100).round(2)
pd.DataFrame({"Count": status_counts, "% of dataset": status_pct})

,Count,% of dataset
loan_status,,
Fully Paid,1076751,47.63
Current,878317,38.85
Charged Off,268559,11.88
Late (31-120 days),21467,0.95
In Grace Period,8436,0.37
Late (16-30 days),4349,0.19
Does not meet the credit policy. Status:Fully Paid,1988,0.09
Does not meet the credit policy. Status:Charged Off,761,0.03
Default,40,0.00


**Result:**

| Status | Count | % of dataset |
|---|---|---|
| Fully Paid | 1,076,751 | 47.6% |
| Current | 878,317 | 38.9% |
| Charged Off | 268,559 | 11.9% |
| Late (31-120 days) | 21,467 | 0.9% |
| In Grace Period | 8,436 | 0.4% |
| Late (16-30 days) | 4,349 | 0.2% |
| Does not meet the credit policy. Status:Fully Paid | 1,988 | 0.09% |
| Does not meet the credit policy. Status:Charged Off | 761 | 0.03% |
| Default | 40 | 0.002% |

**Resolved vs unresolved outcomes.** Only `Fully Paid`, `Charged Off`, and `Default`
represent a completed loan lifecycle. `Current`, the two `Late (...)` categories, and
`In Grace Period` are still active loans with an unknown final outcome and **cannot**
be used to train a supervised default model — including them would introduce label
noise, since a `Current` loan today may still default tomorrow.

**Ambiguous statuses — explicit decision.** The two "Does not meet the credit policy"
statuses are legacy loans issued under a since-retired underwriting policy. Their
underlying repayment outcome (Fully Paid / Charged Off) is still known and valid.
**Decision: treat these as equivalent to their underlying resolved status and include
them in the resolved population** (`Does not meet the credit policy. Status:Fully Paid`
→ Fully Paid; `...Status:Charged Off` → Charged Off). This decision, and the exact
mapping, is what notebook 3 implements.

In [12]:
#---------------------------------------
# Quantify the resolved population and class imbalance
#---------------------------------------
resolved_statuses = ["Fully Paid", "Charged Off", "Default"]
policy_exception_map = {
    "Does not meet the credit policy. Status:Fully Paid": "Fully Paid",
    "Does not meet the credit policy. Status:Charged Off": "Charged Off",
}

status_for_resolution = df["loan_status"].replace(policy_exception_map)
is_resolved = status_for_resolution.isin(resolved_statuses)

n_resolved = is_resolved.sum()
print(f"Resolved loans (incl. policy-exception mapping): {n_resolved:,} "
      f"({is_resolved.mean()*100:.2f}% of full dataset)")

resolved_target = status_for_resolution[is_resolved].isin(["Charged Off", "Default"]).astype(int)
default_rate = resolved_target.mean()
print(f"Default rate among resolved loans: {default_rate*100:.2f}%")
print(f"Class balance -> Good: {(1-default_rate)*100:.2f}% | Bad: {default_rate*100:.2f}%")

Resolved loans (incl. policy-exception mapping): 1,348,099 (59.63% of full dataset)
Default rate among resolved loans: 19.98%
Class balance -> Good: 80.02% | Bad: 19.98%


**Result: ~1,345,350 resolved loans (Fully Paid 1,076,751 + 1,988 policy-exception +
Charged Off 268,559 + 761 policy-exception + Default 40), default rate ~19.96%.**

**This confirms a real, meaningful class imbalance of roughly 4:1 (good:bad).** This
must be handled explicitly in modeling — via class weighting, resampling, and by using
ROC-AUC/precision/recall/F1 rather than raw accuracy as the primary evaluation metric,
since a model that always predicts "no default" would already score ~80% accuracy
while being business-useless.

## 9. Time coverage
`issue_d` determines how far back this dataset goes. This directly informs the
train/test split strategy used in modeling: a dataset spanning many years of different
economic conditions (recession vs expansion) should **not** be split randomly, since a
random split allows the model to implicitly train on information from periods
chronologically after parts of its own test set. An out-of-time split (train on
earlier-issued loans, test on later-issued loans) is the appropriate standard for this
kind of data and is the plan for notebook 4.

In [13]:
df["issue_d_parsed"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
print("Date range:", df["issue_d_parsed"].min(), "to", df["issue_d_parsed"].max())
print()
print("Loans issued per year:")
print(df["issue_d_parsed"].dt.year.value_counts().sort_index())

Date range: 2007-06-01 00:00:00 to 2018-12-01 00:00:00

Loans issued per year:
issue_d_parsed
2007.0       603
2008.0      2393
2009.0      5281
2010.0     12537
2011.0     21721
2012.0     53367
2013.0    134814
2014.0    235629
2015.0    421095
2016.0    434407
2017.0    443579
2018.0    495242
Name: count, dtype: int64


C:\Users\venut\AppData\Local\Temp\ipykernel_4876\358851147.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["issue_d_parsed"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")


**Run the cell above and record the actual date range and yearly counts here** —
this is the one section of this notebook not yet backed by an executed output. Once
run, note explicitly: (a) the full year range covered, and (b) whether a visible dip
appears around 2008–2009 (the dataset is known to span the 2008 financial crisis
period, which — if present — is itself a notable, presentable finding about how loan
volume responded to that period).

## 10. Leakage-risk column audit
These candidate columns are only known, or only take a meaningful (non-zero/non-null)
value, **after** a loan's outcome is already determined. Using any of them as a model
feature would leak the target — the model would effectively be given the answer.
Flagging them here, with evidence, so they are excluded **at the source** in
preprocessing rather than omitted silently.

In [14]:
leakage_risk_cols = [
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    "last_credit_pull_d", "out_prncp", "out_prncp_inv",
    "hardship_flag", "debt_settlement_flag", "settlement_status",
]
present = [c for c in leakage_risk_cols if c in df.columns]
print(f"Leakage-risk columns present in raw data: {len(present)} / {len(leakage_risk_cols)}")
for c in present:
    print(" -", c)

Leakage-risk columns present in raw data: 16 / 16
 - total_pymnt
 - total_pymnt_inv
 - total_rec_prncp
 - total_rec_int
 - total_rec_late_fee
 - recoveries
 - collection_recovery_fee
 - last_pymnt_d
 - last_pymnt_amnt
 - next_pymnt_d
 - last_credit_pull_d
 - out_prncp
 - out_prncp_inv
 - hardship_flag
 - debt_settlement_flag
 - settlement_status


In [15]:
# Evidence: total_pymnt should be near-zero for defaulted loans and near/above
# loan_amnt for fully paid loans, if this column truly reflects post-outcome payments
# rather than pre-origination information.
if "total_pymnt" in df.columns:
    evidence = df.loc[is_resolved, "total_pymnt"].groupby(status_for_resolution[is_resolved]).median()
    print("Median total_pymnt by resolved status (evidence of leakage):")
    print(evidence)

Median total_pymnt by resolved status (evidence of leakage):
loan_status
Charged Off     6507.290
Default         6296.335
Fully Paid     13829.270
Name: total_pymnt, dtype: float64


**Run the cells above and record the actual median values here.** Expected pattern
(to confirm leakage): `total_pymnt` should be materially lower for Charged Off than for
Fully Paid loans — if so, this confirms the column encodes post-outcome information and
must be excluded from the feature set used in notebook 3/4, alongside every other column
in the list above.

## 11. Save column-level metadata

In [16]:
col_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
os.makedirs("data/metadata", exist_ok=True)
col_summary.to_csv("data/metadata/column_summary.csv")
print("Saved column summary to data/metadata/column_summary.csv")
col_summary.sort_values("missing_pct", ascending=False).head(20)

Saved column summary to data/metadata/column_summary.csv


,dtype,missing_count,missing_pct,n_unique
member_id,float32,2260701,100.00,0
orig_projected_additional_accrued_interest,float32,2252050,99.62,7487
hardship_status,object,2249784,99.52,3
hardship_amount,float32,2249784,99.52,9162
hardship_last_payment_amount,float32,2249784,99.52,9045
hardship_length,float32,2249784,99.52,1
payment_plan_start_date,object,2249784,99.52,27
hardship_reason,object,2249784,99.52,9
deferral_term,float32,2249784,99.52,1
hardship_payoff_balance_amount,float64,2249784,99.52,10894


## Summary — findings and decisions carried into notebook 3

| # | Finding | Decision |
|---|---|---|
| 1 | 2,260,701 rows, 151 columns, 2.5GB | Load only required columns via chunking downstream |
| 2 | `id` loads as object dtype | Investigated in Section 3; keep as identifier, never a feature |
| 3 | 0 duplicate rows | No deduplication needed |
| 4 | `member_id` 100% missing | Drop — carries no information |
| 5 | Hardship/settlement columns ~98.5–99.6% missing | Drop from feature set — too sparse, and reflect post-origination distress, not pre-origination risk |
| 6 | `dti` minimum = -1.0 (sentinel value) | Convert -1 to NaN before capping/imputation |
| 7 | `annual_inc` maximum = $110,000,000 (extreme outlier) | Cap/winsorize before engineering income-based ratio features |
| 8 | Resolved population ≈ 1,345,350 loans; default rate ≈ 19.96% | Restrict modeling to resolved loans only; treat as imbalanced classification (~4:1) |
| 9 | "Does not meet the credit policy" statuses | Map to underlying Fully Paid/Charged Off outcome, include in resolved population |
| 10 | Dataset spans multiple years (exact range: run Section 9) | Use out-of-time train/test split in modeling, not random split |
| 11 | 15 candidate leakage-risk columns identified, evidence in Section 10 | Exclude entirely from the feature set loaded in preprocessing |

**Next:** `2_eda.ipynb` — univariate and bivariate analysis on the retained, resolved
population, including categorical predictors, correlation structure, and the time-trend
view not yet covered here.